In [0]:
# Databricks notebook source

# COMMAND ----------

RAW_FILE = "/Volumes/marathos/default/raw/TWO_CENTURIES_OF_UM_RACES 2.csv"

# COMMAND ----------

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_FILE)
)

# COMMAND ----------

print("Rows:", df.count())
print("Columns:", len(df.columns))
df.printSchema()
df.columns

# COMMAND ----------

display(
    df.describe(
        "Year of event",
        "Event number of finishers",
        "Athlete year of birth",
        "Athlete average speed"
    )
)

# COMMAND ----------

from pyspark.sql.functions import col, sum as spark_sum, when

null_df = df.select([
    spark_sum(
        when(col(c).cast("string").isNull() | (col(c).cast("string") == ""), 1)
        .otherwise(0)
    ).alias(c)
    for c in df.columns
])

display(null_df)

# COMMAND ----------

display(
    df.select("Event name")
    .distinct()
    .orderBy("Event name")
)

print("Unique events:", df.select("Event name").distinct().count())

# COMMAND ----------

from pyspark.sql.functions import expr

age_df = (
    df.withColumn("age", expr("`Year of event` - `Athlete year of birth`"))
    .filter(col("age").between(10, 90))
)

display(
    age_df.groupBy("age")
    .count()
    .orderBy("age")
)

# COMMAND ----------

display(
    df.groupBy("Athlete country")
    .count()
    .orderBy(col("count").desc())
    .limit(20)
)

# COMMAND ----------

from pyspark.sql.functions import regexp_extract

units = (
    df.withColumn(
        "dist_unit",
        regexp_extract(col("Event distance/length"), r"([a-z]+)$", 1)
    )
    .withColumn(
        "perf_unit",
        regexp_extract(col("Athlete performance"), r"([a-z]+)$", 1)
    )
)

display(
    units.groupBy("dist_unit", "perf_unit")
    .count()
    .orderBy(col("count").desc())
)

# COMMAND ----------

import plotly.express as px

top_countries = (
    df.groupBy("Athlete country")
    .count()
    .orderBy(col("count").desc())
    .limit(15)
    .toPandas()
)

fig = px.bar(
    top_countries,
    x="Athlete country",
    y="count",
    title="Top 15 countries"
)

fig.show()